# RAG mit DuckDB: Minimalbeispiel

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abuckenhofer/rag-vorlesung/blob/main/RAG_DuckDB_Minimal.ipynb)

Dieses Notebook baut eine vollständige RAG-Anwendung fast nur mit SQL: Wikipedia-Artikel laden, in Chunks zerlegen, Embeddings berechnen, semantisch suchen, mit einem LLM antworten und das Retrieval messen. Python wird nur gebraucht, um das Embedding-Modell und das LLM als SQL-Funktionen in DuckDB zu registrieren.

| Schritt | Teil der Architektur | Werkzeug |
|---|---|---|
| 1 Setup | | Python |
| 2 Daten laden | Index Pipeline | SQL mit `read_json` |
| 3 Chunking | Index Pipeline | SQL |
| 4 Embeddings | Index Pipeline | SQL mit `embed()` |
| 5 Semantische Suche | Vektorspeicher | SQL |
| 6 RAG | RAG Pipeline | SQL mit `llm()` |
| 7 Evaluation | | SQL |

**Voraussetzungen:** Google Colab oder ein lokales Jupyter. Für das LLM in Schritt 6 gibt es zwei Zugänge, gewählt mit dem Schalter `LLM_ZUGANG` in der ersten Code-Zelle:

| `LLM_ZUGANG` | Was wird gebraucht? | Läuft wo? |
|---|---|---|
| `"colab"` (Standard) | nichts, Gemini ist in Colab eingebaut, mit monatlichem Kontingent | nur in Colab |
| `"account"` | eigener kostenloser Gemini-API-Key von https://aistudio.google.com/apikey, ohne Kreditkarte, am besten mit privatem Google-Konto (Hochschulkonten sperren den Zugriff oft) | Colab (Secret `GEMINI`) und lokal (Umgebungsvariable `GEMINI_API_KEY`) |

Die Schritte 1 bis 5 und die Recall-Messung in Schritt 7 brauchen kein LLM.

**Lokal mit uv:** `uv init rag && cd rag && uv add duckdb sentence-transformers google-genai jupyterlab && uv run jupyter lab`

## 1 Setup

**Konfiguration:** Hier werden der Zugang zum LLM und die Modelle festgelegt. Nach einer Änderung die Setup-Zelle und alle folgenden Zellen erneut ausführen.

In [ ]:
LLM_ZUGANG = "colab"        # "colab": Gemini über Colab ohne Key | "account": eigener Gemini-API-Key
LLM_MODELL = "gemini-3.5-flash-lite"  # nur für "account", aktuelle IDs: https://ai.google.dev/models
EMBEDDING_MODELL = "sentence-transformers/all-mpnet-base-v2"  # 768 Dimensionen, max. 384 Tokens

Installation der drei benötigten Pakete. Nur DuckDB wird aktualisiert. `sentence-transformers` und `google-genai` sind in Colab vorinstalliert, ein Update würde Pakete ändern, von denen Colab selbst abhängt.

In [ ]:
%pip install -q -U duckdb
%pip install -q sentence-transformers google-genai

Diese Zelle ist der einzige Python-Code des Notebooks. Sie registriert zwei Funktionen in DuckDB:

- `embed(text)` wandelt Text mit dem Open-Source-Modell `all-mpnet-base-v2` in einen Vektor `FLOAT[768]` um. Das Modell wird beim ersten Lauf geladen (ca. 400 MB).
- `llm(system, prompt)` schickt Anweisungen und Prompt an Gemini und liefert die Antwort. Mit `"account"` gehen die Anweisungen getrennt als System-Prompt an das Modell. Colab AI kennt keinen System-Prompt, dort werden die Anweisungen vor den Prompt gestellt. Welche Modelle Colab AI anbietet, zeigt `ai.list_models()`.

Die Datenbank liegt nur im Arbeitsspeicher. Jeder Lauf dieser Zelle beginnt daher mit einer leeren Datenbank, danach die folgenden Zellen erneut ausführen. Alle anderen Zellen lassen sich beliebig oft wiederholen.

Der eigene User-Agent identifiziert die Anfragen an Wikipedia, wie es die Wikimedia-Richtlinie verlangt.

In [ ]:
import os, duckdb
from functools import cache
from sentence_transformers import SentenceTransformer

if LLM_ZUGANG == "colab":
    from google.colab import ai  # nur in Colab verfügbar
    llm = lambda system, prompt: str(ai.generate_text(system + "\n\n" + prompt))
elif LLM_ZUGANG == "account":
    from google import genai
    try:  # Colab: Key aus dem Secret GEMINI, lokal: Umgebungsvariable GEMINI_API_KEY
        from google.colab import userdata
        os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI")
    except Exception:
        pass
    gemini = cache(genai.Client)  # ein Client für alle Aufrufe, erst beim ersten LLM-Aufruf erzeugt
    llm = lambda system, prompt: gemini().models.generate_content(
        model=LLM_MODELL, contents=prompt, config={"system_instruction": system}).text
else:
    raise ValueError('LLM_ZUGANG muss "colab" oder "account" sein')

modell = SentenceTransformer(EMBEDDING_MODELL)

# In-Memory-Datenbank: jeder Lauf dieser Zelle beginnt mit einer leeren Datenbank
con = duckdb.connect(config={"custom_user_agent": "RAG-Vorlesung (https://github.com/abuckenhofer/rag-vorlesung)"})

con.create_function("embed", lambda text: modell.encode(text, normalize_embeddings=True).tolist(),
                    ["VARCHAR"], "FLOAT[768]")
con.create_function("llm", llm, ["VARCHAR", "VARCHAR"], "VARCHAR", side_effects=True)

## 2 Daten laden (Index Pipeline)

DuckDB liest die Wikipedia-API direkt als JSON. Die API liefert den Artikel als reinen Text (`explaintext`), HTML muss also nicht bereinigt werden. Weitere Artikel lassen sich einfach an die Liste anhängen.

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE dokumente AS
SELECT p.title AS titel,
       'https://en.wikipedia.org/wiki/' || replace(p.title, ' ', '_') AS url,
       p.extract AS text
FROM (SELECT unnest(query.pages) AS p
      FROM read_json([
        'https://en.wikipedia.org/w/api.php?action=query&format=json&formatversion=2&prop=extracts&explaintext=1&titles=Retrieval-augmented_generation',
        'https://en.wikipedia.org/w/api.php?action=query&format=json&formatversion=2&prop=extracts&explaintext=1&titles=Vector_database',
        'https://en.wikipedia.org/w/api.php?action=query&format=json&formatversion=2&prop=extracts&explaintext=1&titles=Hierarchical_navigable_small_world',
        'https://en.wikipedia.org/w/api.php?action=query&format=json&formatversion=2&prop=extracts&explaintext=1&titles=Prompt_engineering'
      ]))
""")

con.sql("SELECT titel, length(text) AS zeichen, url FROM dokumente")

## 3 Chunking

Strategie: ein Absatz ergibt einen Chunk. Überschriften im Format `== Titel ==` werden nicht zu eigenen Chunks, sondern als Spalte `abschnitt` an jeden folgenden Absatz gehängt. Das erledigt die Fensterfunktion `last_value(... IGNORE NULLS)`. Abschnitte ohne Fließtext wie *See also* oder *References* werden herausgefiltert.

Die Spalte `emb` bleibt zunächst leer und wird in Schritt 4 befüllt.

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE chunks AS
WITH zeilen AS (
    SELECT titel, url, nr, trim(string_split(text, chr(10))[nr]) AS zeile
    FROM dokumente, range(1, len(string_split(text, chr(10))) + 1) AS r(nr)
), mit_abschnitt AS (
    SELECT *,
           coalesce(last_value(CASE WHEN zeile LIKE '==%' THEN trim(zeile, '= ') END IGNORE NULLS)
                    OVER (PARTITION BY titel ORDER BY nr), 'Einleitung') AS abschnitt
    FROM zeilen
)
SELECT row_number() OVER (ORDER BY titel, nr) AS id,
       titel, abschnitt, url, zeile AS text,
       NULL::FLOAT[768] AS emb
FROM mit_abschnitt
WHERE zeile <> ''
  AND zeile NOT LIKE '==%'
  AND abschnitt NOT IN ('See also', 'References', 'External links', 'Further reading', 'Notes')
""")

con.sql("SELECT id, titel, abschnitt, left(text, 80) AS text FROM chunks LIMIT 10")

**Prüfung der Chunk-Länge:** `all-mpnet-base-v2` verarbeitet maximal 384 Tokens, bei englischem Text rund 1.500 Zeichen. Was darüber hinausgeht, schneidet das Modell ohne Warnung ab. Die Abfrage zeigt, wie viele Chunks betroffen sind.

In [ ]:
con.sql("""
SELECT count(*)                                AS chunks,
       round(median(length(text)))             AS median_zeichen,
       max(length(text))                       AS max_zeichen,
       count(*) FILTER (WHERE length(text) > 1500) AS zu_lang
FROM chunks
""")

## 4 Embeddings (Index Pipeline)

Titel und Abschnitt werden mit eingebettet. So behält ein Absatz wie "It was introduced in 2020" seinen Kontext. Die Vektoren sind auf Länge 1 normiert. Dauer auf einer CPU: etwa eine Minute.

In [ ]:
con.sql("UPDATE chunks SET emb = embed(titel || ' - ' || abschnitt || ': ' || text)")

con.sql("SELECT id, abschnitt, emb[1:4] AS emb_anfang, len(emb) AS dimensionen FROM chunks LIMIT 5")

## 5 Semantische Suche (Vektorspeicher)

Das Makro `suche(frage, k)` bettet die Frage mit demselben Modell ein und sortiert alle Chunks nach Kosinus-Distanz. `array_cosine_distance` entspricht dem Operator `<=>` in pgvector: 0 bedeutet gleiche Richtung.

Ohne Index ist die Suche exakt und vergleicht die Frage mit jedem Chunk. Für große Datenmengen bietet die DuckDB-Extension `vss` einen HNSW-Index.

In [ ]:
con.sql("""
CREATE OR REPLACE MACRO suche(frage, k := 5) AS TABLE
    SELECT id, titel, abschnitt, url, text,
           array_cosine_distance(emb, embed(frage)) AS distanz
    FROM chunks
    ORDER BY distanz
    LIMIT k
""")

con.sql("SELECT titel, abschnitt, round(distanz, 3) AS distanz, left(text, 80) AS text FROM suche('Does RAG prevent hallucinations?')")

## 6 RAG: Retrieval, Augmentation, Generation (RAG Pipeline)

Drei Makros bilden die RAG Pipeline:

- `system_prompt()` enthält die Anweisungen: nur mit dem Kontext antworten, Quellen nennen und "Das weiß ich nicht" ist erlaubt.
- `prompt(frage, k)` baut den Kontext aus den Top-k Chunks, jeder mit Quellen-ID.
- `rag(frage, k)` schickt beides an das LLM.

Anweisungen und Daten sind dabei getrennt: Die Anweisungen stehen im System-Prompt, der Kontext im Prompt des Nutzers.

In [ ]:
con.sql("""
CREATE OR REPLACE MACRO system_prompt() AS
    'Beantworte die Frage ausschließlich mit Informationen aus dem Kontext und nenne zu jeder Aussage '
    || 'die Quelle, z.B. [1]. Du darfst Informationen aus mehreren Abschnitten zusammenfassen. '
    || 'Antworte nur dann mit "Das weiß ich nicht.", wenn der Kontext keine relevanten Informationen '
    || 'zur Frage enthält. Antworte auf Deutsch.'
""")

con.sql("""
CREATE OR REPLACE MACRO prompt(frage, k := 5) AS (
    SELECT '<kontext>' || chr(10)
           || string_agg(format('[{}] {} - {}: {}', nr, titel, abschnitt, text), chr(10) ORDER BY nr)
           || chr(10) || '</kontext>' || chr(10) || 'Frage: ' || frage
    FROM (SELECT row_number() OVER (ORDER BY distanz) AS nr, * FROM suche(frage, k))
)
""")

con.sql("CREATE OR REPLACE MACRO rag(frage, k := 5) AS llm(system_prompt(), prompt(frage, k))")

print(con.sql("SELECT prompt('Does RAG prevent hallucinations?', 3)").fetchall()[0][0])

Jetzt mit dem LLM:

In [ ]:
print(con.sql("SELECT rag('Does RAG prevent hallucinations?')").fetchall()[0][0])

Gegenprobe mit einer Frage, die der Kontext nicht beantwortet. Das LLM kennt die Antwort aus dem Training, soll sie aber laut Anweisung nicht verwenden:

In [ ]:
print(con.sql("SELECT rag('Who won the FIFA World Cup 2022?')").fetchall()[0][0])

## 7 Evaluation

Eine RAG-Antwort kann an zwei Stellen scheitern: Die Suche findet den passenden Chunk nicht (Retrieval), oder das LLM nutzt ihn nicht (Generation). Die Evaluation prüft beide Stufen getrennt, damit klar ist, wo nachgebessert werden muss.

**Testset:** Jede Frage hat eine erwartete Quelle und einen Beleg, also ein Wort oder eine kurze Textstelle, die im passenden Chunk vorkommen muss. Ein Treffer zählt nur, wenn ein Chunk mit diesem Beleg gefunden wird. Ein beliebiger Chunk der richtigen Seite reicht nicht, denn bei vier Artikeln stimmt die Seite fast immer.

Die Spalte `beleg_vorhanden` prüft das Testset selbst: Steht dort `false`, kommt der Beleg in keinem Chunk vor, und Frage oder Beleg müssen angepasst werden.

Die Spalte `rang` zeigt, auf welchem Platz der Suche der erste passende Chunk steht. Leer heißt: nicht unter den Top 10.

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE testset AS
SELECT * FROM (VALUES
    ('Does RAG prevent hallucinations?',                                   'Retrieval-augmented generation',     'does not prevent hallucinations'),
    ('Why are documents split into chunks?',                               'Retrieval-augmented generation',     'chunk'),
    ('What is the role of layers in HNSW?',                                'Hierarchical navigable small world', 'layer'),
    ('What is chain-of-thought prompting?',                                'Prompt engineering',                 'chain-of-thought'),
    ('Which algorithms are used for approximate nearest neighbor search?', 'Vector database',                    'nearest neighbo')
) AS t(frage, quelle, beleg)
""")

# Auf welchem Platz steht der erste Chunk der erwarteten Quelle mit dem Beleg? NULL = nicht unter den Top-k
con.sql("""
CREATE OR REPLACE MACRO rang(frage, quelle, beleg, k := 10) AS (
    SELECT min(nr)
    FROM (SELECT row_number() OVER (ORDER BY distanz) AS nr, titel, text FROM suche(frage, k))
    WHERE titel = quelle AND text ILIKE '%' || beleg || '%'
)
""")

con.sql("CREATE OR REPLACE MACRO treffer(frage, quelle, beleg, k) AS coalesce(rang(frage, quelle, beleg, k := k) <= k, false)")

con.sql("""
SELECT frage, beleg,
       EXISTS (SELECT 1 FROM chunks c
               WHERE c.titel = t.quelle AND c.text ILIKE '%' || t.beleg || '%') AS beleg_vorhanden,
       rang(frage, quelle, beleg) AS rang
FROM testset t
""")

### Retrieval: Recall@k

Recall@k ist der Anteil der Fragen, bei denen der Beleg unter den Top-k Chunks ist, also `rang <= k`. Der Mean Reciprocal Rank (MRR) fasst die Plätze in einer Zahl zusammen: Platz 1 zählt 1, Platz 2 zählt 0,5, Platz 3 zählt 0,33, nicht gefunden zählt 0. Beides braucht kein LLM, ist schnell und reproduzierbar. So lässt sich prüfen, ob eine Änderung an Chunking oder Embedding-Modell wirklich hilft. In der Praxis gehören 20 bis 50 echte Fragen ins Testset.

In [ ]:
con.sql("""
SELECT round(avg(treffer(frage, quelle, beleg, 1)::INT), 2) AS recall_at_1,
       round(avg(treffer(frage, quelle, beleg, 3)::INT), 2) AS recall_at_3,
       round(avg(treffer(frage, quelle, beleg, 5)::INT), 2) AS recall_at_5,
       round(avg(coalesce(1 / rang(frage, quelle, beleg), 0)), 2) AS mrr
FROM testset
""")

### Generation: Wird die Frage beantwortet?

Jetzt beantwortet das LLM alle Fragen des Testsets. Die Antworten werden in einer Tabelle gespeichert, damit die Auswertung das LLM nicht erneut aufruft.

Die Diagnose kombiniert beide Stufen. Geprüft wird mit k = 5, weil `rag()` dem LLM die Top 5 Chunks gibt:

| Beleg im Kontext | Beantwortet | Diagnose |
|---|---|---|
| ja | ja | ok |
| nein | nein | Retrieval verbessern: Chunking, Embedding-Modell, Hybrid Search |
| ja | nein | Kontext da, keine Antwort: Prompt, Modell oder Testfrage prüfen |
| nein | ja | Antwort ohne Beleg: Antwort prüfen |

Die Prüfung über ein Stichwort ist eine Heuristik und irrt in beide Richtungen. Ein Chunk kann das Stichwort enthalten, ohne die Frage zu beantworten. Umgekehrt kann die Antwort in einer anderen Quelle stehen als erwartet oder anders formuliert sein. Ein Beispiel liefert die Frage zu Nearest Neighbor: Die passenden Chunks stammen aus dem HNSW-Artikel, das Testset erwartet aber "Vector database". Die Diagnose zeigt daher, wo ein Mensch nachsehen sollte, und ist kein Urteil.

"Beantwortet" heißt nur, dass das LLM nicht "Das weiß ich nicht" sagt. Ob die Antwort inhaltlich stimmt, prüft dieser einfache Test nicht. Dafür braucht es einen Menschen oder ein zweites LLM als Bewerter (LLM-as-a-Judge).

In [ ]:
con.sql("CREATE OR REPLACE TABLE antworten AS SELECT frage, rag(frage) AS antwort FROM testset")

con.sql("""
SELECT frage, beleg_im_kontext, beantwortet,
       CASE WHEN beleg_im_kontext AND beantwortet THEN 'ok'
            WHEN NOT beleg_im_kontext AND NOT beantwortet THEN 'Retrieval verbessern'
            WHEN beleg_im_kontext THEN 'Kontext da, keine Antwort: Prompt, Modell oder Testfrage prüfen'
            ELSE 'Antwort ohne Beleg: Antwort prüfen'
       END AS diagnose,
       antwort
FROM (SELECT t.frage,
             treffer(t.frage, t.quelle, t.beleg, 5)    AS beleg_im_kontext,
             a.antwort NOT ILIKE '%weiß ich nicht%'     AS beantwortet,
             a.antwort
      FROM testset t JOIN antworten a USING (frage))
""")

## Nächste Schritte

- Eigene Dokumente laden, z.B. mit `read_text('*.md')` oder `read_csv`
- Chunking variieren und den Recall vergleichen
- Hybrid Search: Volltextsuche mit der DuckDB-Extension `fts` und semantische Suche per Reciprocal Rank Fusion kombinieren
- HNSW-Index mit der Extension `vss` für große Datenmengen